# Paired synthetic secretion analysis with the shared estimator

This comparator is a complete notebook workflow using the same checked OT-054 estimator, engineering variants, keyed joint perturbations and complete report as the CLI. It provides no human usability, biological validity or AOMM/SAAM II equivalence evidence. An experienced analyst must qualify the notebook before human comparison.

Inputs are explicitly invented observations. Total secretion includes basal over exactly 0–120 minutes, in pmol. Change is after minus before. C-peptide AUC and glucose context are separate. Required failures and all perturbations remain visible.

Run the canonical example command before using the default paths. Choose a fresh output directory for each study. In Jupyter, edit the paths in the next cell if needed. The technical cell runner injects these same parameters. Jupyter UI behavior is a separate setup check.

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "experiments/observation_analysis").is_dir()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
PARAMETERS = globals().get("PARAMETERS", {})
PACKAGE_FILE = Path(
    PARAMETERS.get("PACKAGE_FILE", ROOT / "results/ot056-notebook-input/package.json")
)
CSV_FILE = Path(PARAMETERS.get("CSV_FILE", ROOT / "results/ot056-notebook-input/observations.csv"))
SETTINGS_FILE = Path(
    PARAMETERS.get("SETTINGS_FILE", ROOT / "results/ot056-notebook-input/settings.json")
)
OUTPUT = Path(PARAMETERS.get("OUTPUT_DIRECTORY", ROOT / "results/ot056-notebook-output"))

from experiments.catalogue.storage import dump, fresh  # noqa: E402 -- root path initialized above

if "OUTPUT_DIRECTORY" not in PARAMETERS:
    OUTPUT = fresh(OUTPUT)

## Freeze the actual input and protocol before evaluation

The shared freezer binds raw source bytes, explicit visit order, fixed kinetics and volume, all required variants, invented error assumptions, source/runtime and model ancestry. It performs no demographic normalization or outcome-driven tuning. This notebook retains the same opaque source snapshots as the CLI. Archived Python is never imported.

In [ ]:
from experiments.observation_analysis.plan import freeze, load_plan

freeze(PACKAGE_FILE, CSV_FILE, SETTINGS_FILE, OUTPUT / "frozen")
plan, package, raw = load_plan(OUTPUT / "frozen", check_runtime=True)
assert plan["kind"] == "synthetic" and plan["human_execution_authorized"] is False
print("Method:", plan["method"])
print("Ordered visits:", plan["settings"]["visit_order"])
print("Required variants:", [item["id"] for item in plan["settings"]["variants"]])
print("Planned participants:", len(package["participants"]))

## Run the same estimator and full conditional uncertainty procedure

This calls the checked shared calculation directly, without invoking the CLI. It retains the entire roster, invalid/censored observations, negative-rate diagnostics, every required variant and every perturbation. A failed draw is not removed to manufacture an interval. Retained baseline observations initialize each visit under the frozen conventions.

In [ ]:
from experiments.observation_analysis.analysis import evaluate, report_from_attempts
from experiments.observation_analysis.report import render

attempts = evaluate(plan["settings"], package, raw["observations.csv"])
report = report_from_attempts(plan, attempts)
report["execution_status"] = "completed"
dump(OUTPUT / "attempts.json", attempts)
dump(OUTPUT / "report.json", report)
report_markdown = render(report)
(OUTPUT / "report.md").write_bytes(report_markdown)
print("Coverage:", report["coverage"])
for participant_id, detail in sorted(report["people"].items()):
    print(participant_id, detail["classification"], detail["status"])

Task 2 has its own fixed same-estimator comparator in `same-estimator-sampling.ipynb`. Its workflow freezes development selection and removes raw observations before each shared calculation. Human qualification and timed sessions remain separate.

In [ ]:
try:
    from IPython.display import Markdown, display
except ImportError:
    print(report_markdown.decode())
else:
    display(Markdown(report_markdown.decode()))